# Silver PSGC place

## Purpose

Create the reusable official PSGC place hierarchy for population reconciliation and later geographic matching. One output row represents one Bronze PSGC place in the selected source snapshot. The natural key is `psgc_code` plus `COALESCE(psgc_version, source_snapshot_id)`. Raw and standardized names remain side by side. Unresolved geographic levels remain visible rather than being filtered.


## 1. Confirm the Bronze safety gate

Silver reads the newest terminal audit state, not an older successful event. It also requires positive loaded rows, snapshot identity, and a latest Bronze validation run with no blocking failure. This reads only compact audit and validation metadata.


In [ ]:
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `02-silver`;

CREATE OR REPLACE TEMPORARY VIEW latest_psgc_load AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY table_name
            ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
        ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'psgc'
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
)
WHERE terminal_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')) = 1
    AND COUNT_IF(snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> '') = 1
    AND COUNT_IF(rows_loaded > 0) = 1,
    'PSGC Bronze safety gate failed: latest audit must be successful, identified, and non-empty'
)
FROM latest_psgc_load;

CREATE OR REPLACE TEMPORARY VIEW latest_psgc_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT
        *,
        DENSE_RANK() OVER (ORDER BY run_ts DESC, run_id DESC) AS validation_rank
    FROM `04-validation`.dq_results
    WHERE table_name = 'psgc'
      AND snapshot_id = (SELECT snapshot_id FROM latest_psgc_load)
)
WHERE validation_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'PSGC Bronze safety gate failed: latest blocking validation did not pass'
)
FROM latest_psgc_validation;


## 2. Standardize official place identity

The selected PSGC snapshot is the only business input. Name normalization changes case and repeated whitespace only. Official geographic-level values map to a small place-type contract. Blank or unfamiliar levels become `UNASSIGNED`; their source rows remain in the output. A deterministic source-row identity uses source lineage and snapshot identity.


In [ ]:
CREATE OR REPLACE TEMPORARY VIEW psgc_place_base AS
SELECT
    TRIM(psgc_code) AS psgc_code,
    name AS place_name_raw,
    REGEXP_REPLACE(UPPER(TRIM(name)), '\\s+', ' ') AS place_name_standardized,
    geographic_level AS geographic_level_raw,
    CASE UPPER(TRIM(geographic_level))
        WHEN 'REG' THEN 'REGION'
        WHEN 'PROV' THEN 'PROVINCE'
        WHEN 'CITY' THEN 'CITY'
        WHEN 'MUN' THEN 'MUNICIPALITY'
        WHEN 'SUBMUN' THEN 'SUBMUNICIPALITY'
        WHEN 'BGY' THEN 'BARANGAY'
        ELSE 'UNASSIGNED'
    END AS place_type,
    TRY_CAST(NULLIF(TRIM(population_2024), '') AS BIGINT) AS psgc_population_crosscheck,
    NULLIF(TRIM(source_release), '') AS psgc_version,
    _source_snapshot_id AS source_snapshot_id,
    _ingest_run_id AS source_ingest_run_id,
    _ingested_at AS source_load_ts,
    source_file,
    source_sheet,
    source_row_number,
    SHA2(CONCAT_WS('|', _source_snapshot_id, source_file, source_sheet, source_row_number), 256)
        AS source_row_identity
FROM `01-bronze`.psgc;


## 3. Build hierarchy from verified PSGC rows

PSGC codes locate candidate ancestors, but names and parent codes come only from rows that exist in the selected PSGC snapshot. A missing parent remains null. Repeated place names are expected and never treated as unique without code and hierarchy context. The output grain remains one source PSGC row.


In [ ]:
CREATE OR REPLACE TABLE `02-silver`.silver_psgc_place
USING DELTA
COMMENT 'Official PSGC place hierarchy for the selected source snapshot'
AS
WITH hierarchy AS (
    SELECT
        child.*,
        region.psgc_code AS region_code,
        region.place_name_raw AS region_name,
        province.psgc_code AS province_code,
        province.place_name_raw AS province_name,
        locality.psgc_code AS city_municipality_code,
        locality.place_name_raw AS city_municipality_name
    FROM psgc_place_base AS child
    LEFT JOIN psgc_place_base AS region
        ON region.psgc_code = CONCAT(SUBSTRING(child.psgc_code, 1, 2), '00000000')
       AND region.place_type = 'REGION'
       AND region.source_snapshot_id = child.source_snapshot_id
    LEFT JOIN psgc_place_base AS province
        ON province.psgc_code = CONCAT(SUBSTRING(child.psgc_code, 1, 5), '00000')
       AND province.place_type = 'PROVINCE'
       AND province.source_snapshot_id = child.source_snapshot_id
    LEFT JOIN psgc_place_base AS locality
        ON locality.psgc_code = CONCAT(SUBSTRING(child.psgc_code, 1, 7), '000')
       AND locality.place_type IN ('CITY', 'MUNICIPALITY', 'SUBMUNICIPALITY')
       AND locality.source_snapshot_id = child.source_snapshot_id
)
SELECT
    psgc_code,
    place_name_raw,
    place_name_standardized,
    geographic_level_raw,
    place_type,
    region_code,
    region_name,
    province_code,
    province_name,
    city_municipality_code,
    city_municipality_name,
    psgc_population_crosscheck,
    psgc_version,
    source_snapshot_id,
    source_ingest_run_id,
    SHA2(CONCAT_WS('|', source_snapshot_id, COALESCE(psgc_version, 'NO_PUBLISHER_VERSION'), 'silver-psgc-place-v1'), 256) AS run_id,
    source_row_identity,
    source_load_ts,
    CASE
        WHEN place_type = 'UNASSIGNED' THEN 'UNASSIGNED_GEOGRAPHIC_LEVEL'
        WHEN place_type <> 'REGION' AND region_code IS NULL THEN 'MISSING_VERIFIED_REGION'
        WHEN place_type = 'BARANGAY' AND city_municipality_code IS NULL THEN 'MISSING_VERIFIED_LOCALITY'
        ELSE 'VALID_HIERARCHY'
    END AS hierarchy_status,
    'silver-psgc-place-v1' AS transformation_rule_version
FROM hierarchy;


# Summary

Created `02-silver.silver_psgc_place` at one official PSGC place per selected version or snapshot. The natural key is PSGC code plus publisher version when available, otherwise source snapshot ID. The table retains raw names, conservative standardized names, verified hierarchy, source-row lineage, Bronze ingest identity, deterministic Silver run identity, and rule version. The final query reports row accounting and hierarchy status. Requires Databricks execution. Next dependency: population place reconciliation.


In [ ]:
SELECT
    source_snapshot_id,
    COALESCE(psgc_version, 'NO_PUBLISHER_VERSION') AS psgc_version,
    place_type,
    hierarchy_status,
    COUNT(*) AS place_rows
FROM `02-silver`.silver_psgc_place
GROUP BY source_snapshot_id, COALESCE(psgc_version, 'NO_PUBLISHER_VERSION'), place_type, hierarchy_status
ORDER BY place_type, hierarchy_status;
